# Praktikum Minggu 7 — Heart Classification

## Informasi Dataset

- Source: UCI Repository, berdasarkan modul praktikum.
- Attributes: 13 fitur.
- Classes: 2 kelas, yaitu Absence dan Presence.
- Number of data: 270 baris.
  - Absence: 150 data.
  - Presence: 120 data.

Absence menunjukkan tidak adanya penyakit jantung,
sedangkan Presence menunjukkan adanya penyakit jantung.

Dataset memiliki 14 kolom: 13 kolom fitur dan 1 kolom kelas.

In [29]:
import pandas as pd

nama_kolom = [
    "Age", "Sex", "ChestPain", "RestingBP",
    "Cholesterol", "FastingBS", "RestingECG",
    "MaxHR", "ExerciseAngina", "Oldpeak",
    "Slope", "MajorVessels", "Thal", "Class"
]

dataset = pd.read_csv(
    "heart.csv",
    header=None,
    names=nama_kolom
)

print("Dataset Heart:")
print(dataset)

print("\nSource: UCI Repository (berdasarkan modul praktikum)")
print("Attributes:", dataset.drop(columns="Class").shape[1])
print("Classes:", dataset["Class"].nunique())
print("Number of data:", len(dataset))

print("\nJumlah data setiap kelas:")
print(dataset["Class"].value_counts())

print("\nMissing values setiap kolom:")
print(dataset.isna().sum())

# Memastikan dataset sesuai deskripsi modul
assert dataset.shape == (270, 14)
assert dataset["Class"].value_counts().to_dict() == {
    "Absence": 150,
    "Presence": 120
}

print("\nPemeriksaan berhasil: dataset sesuai deskripsi modul.")

Dataset Heart:
     Age     Sex         ChestPain  RestingBP  Cholesterol  FastingBS  \
0     70    male      asymptomatic        130          322          0   
1     67  female  non_anginal_pain        115          564          0   
2     57    male   atypical_angina        124          261          0   
3     64    male      asymptomatic        128          263          0   
4     74  female   atypical_angina        120          269          0   
..   ...     ...               ...        ...          ...        ...   
265   52    male  non_anginal_pain        172          199          1   
266   44    male   atypical_angina        120          263          0   
267   56  female   atypical_angina        140          294          0   
268   57    male      asymptomatic        140          192          0   
269   67    male      asymptomatic        160          286          0   

                  RestingECG  MaxHR ExerciseAngina  Oldpeak      Slope  \
0    ventricular_hypertrophy    10

## Nomor 1 — Konversi Data Kategori Menjadi Numerik
Mengubah fitur kategori dan label kelas menjadi angka agar dapat
digunakan untuk klasifikasi k-NN dan Decision Tree.

In [30]:
dataset_numerik = dataset.copy()

mapping_kategori = {
    "Sex": {
        "female": 0,
        "male": 1
    },
    "ChestPain": {
        "typical_angina": 0,
        "atypical_angina": 1,
        "non_anginal_pain": 2,
        "asymptomatic": 3
    },
    "RestingECG": {
        "normal": 0,
        "wave_abnormality": 1,
        "ventricular_hypertrophy": 2
    },
    "ExerciseAngina": {
        "no": 0,
        "yes": 1
    },
    "Slope": {
        "upsloping": 0,
        "flat": 1,
        "downsloping": 2
    },
    "Thal": {
        "normal": 0,
        "fixed_defect": 1,
        "reversable_defect": 2
    },
    "Class": {
        "Absence": 0,
        "Presence": 1
    }
}

for kolom, mapping in mapping_kategori.items():
    dataset_numerik[kolom] = dataset_numerik[kolom].map(mapping)

# Pastikan seluruh kategori berhasil dikonversi
assert not dataset_numerik.isna().any().any(), (
    "Ada nilai kosong atau kategori yang belum tercakup dalam mapping."
)

# Pisahkan fitur dan label
fitur_data = dataset_numerik.drop(columns="Class")
label = dataset_numerik["Class"]

print("Mapping kategori:")
for kolom, mapping in mapping_kategori.items():
    print(f"{kolom}: {mapping}")

print("\nDataset setelah konversi:")
print(dataset_numerik)

print("\nTipe data setiap kolom:")
print(dataset_numerik.dtypes)

print("\nJumlah data:", len(dataset_numerik))
print("Jumlah fitur:", fitur_data.shape[1])

print("\nJumlah data setiap kelas (0 = Absence, 1 = Presence):")
print(label.value_counts().sort_index())

Mapping kategori:
Sex: {'female': 0, 'male': 1}
ChestPain: {'typical_angina': 0, 'atypical_angina': 1, 'non_anginal_pain': 2, 'asymptomatic': 3}
RestingECG: {'normal': 0, 'wave_abnormality': 1, 'ventricular_hypertrophy': 2}
ExerciseAngina: {'no': 0, 'yes': 1}
Slope: {'upsloping': 0, 'flat': 1, 'downsloping': 2}
Thal: {'normal': 0, 'fixed_defect': 1, 'reversable_defect': 2}
Class: {'Absence': 0, 'Presence': 1}

Dataset setelah konversi:
     Age  Sex  ChestPain  RestingBP  Cholesterol  FastingBS  RestingECG  \
0     70    1          3        130          322          0           2   
1     67    0          2        115          564          0           2   
2     57    1          1        124          261          0           0   
3     64    1          3        128          263          0           0   
4     74    0          1        120          269          0           2   
..   ...  ...        ...        ...          ...        ...         ...   
265   52    1          2        172

## Nomor 2 — Normalisasi Data

Menyiapkan normalisasi Min-Max 0–1, Z-Score, dan Sigmoidal.
Normalisasi hanya diterapkan pada 13 fitur, tanpa kolom Class.

Dalam evaluasi, parameter normalisasi dihitung dari bagian training
setiap fold atau percobaan LOO, lalu diterapkan pada bagian test.

In [31]:
import numpy as np 

def normalisasi_data(data_train, data_test, metode):
    if metode == "Min-Max":
        minimum = data_train.min()
        maksimum = data_train.max()
        rentang = (maksimum - minimum).replace(0, 1)

        train_normal = (data_train - minimum) / rentang
        test_normal = (data_test - minimum) / rentang

        parameter = pd.DataFrame({
            "Minimum": minimum,
            "Maksimum": maksimum
        })

        return train_normal, test_normal, parameter
    elif metode in ["Z-Score", "Sigmoidal"]:
        mean = data_train.mean()
        std = data_train.std(ddof=1)
        pembagi = std.replace(0, 1)

        z_train = (data_train - mean) / pembagi
        z_test = (data_test - mean) / pembagi

        if metode == "Z-Score":
            train_normal = z_train
            test_normal = z_test
        else:
            # tanh(z/2) setara dengan rumus Sigmoidal pada modul
            train_normal = np.tanh(z_train / 2)
            test_normal = np.tanh(z_test / 2)

        parameter = pd.DataFrame({
            "Mean": mean,
            "Standar Deviasi": std
        })

    else:
        raise ValueError("Metode normalisasi tidak dikenal.")

    return train_normal, test_normal, parameter


metode_normalisasi = ["Min-Max", "Z-Score", "Sigmoidal"]

print("Metode normalisasi yang disiapkan:")
for metode in metode_normalisasi:
    print("-", metode)

Metode normalisasi yang disiapkan:
- Min-Max
- Z-Score
- Sigmoidal


In [32]:
from sklearn.model_selection import KFold

contoh_kfold = KFold(
    n_splits=4,
    shuffle=True,
    random_state=42
)

train_idx, test_idx = next(contoh_kfold.split(fitur_data))

contoh_train = fitur_data.iloc[train_idx].copy()
contoh_test = fitur_data.iloc[test_idx].copy()

print("Contoh pembagian: fold pertama K-Fold 4")
print("Jumlah training:", len(contoh_train))
print("Jumlah test:", len(contoh_test))

for metode in metode_normalisasi:
    train_normal, test_normal, parameter = normalisasi_data(
        contoh_train, contoh_test, metode
    )

    print(f"\n=== Normalisasi {metode} ===")

    print("\nParameter dari training:")
    print(parameter.round(4))

    print("\nHasil normalisasi training (5 baris pertama):")
    print(train_normal.head().round(4))

    print("\nHasil normalisasi test (5 baris pertama):")
    print(test_normal.head().round(4))

    print("\nRentang hasil normalisasi training:")
    print(pd.DataFrame({
        "Minimum": train_normal.min(),
        "Maksimum": train_normal.max()
    }).round(4))

    assert np.isfinite(train_normal.to_numpy()).all()
    assert np.isfinite(test_normal.to_numpy()).all()

Contoh pembagian: fold pertama K-Fold 4
Jumlah training: 202
Jumlah test: 68

=== Normalisasi Min-Max ===

Parameter dari training:
                Minimum  Maksimum
Age                29.0      76.0
Sex                 0.0       1.0
ChestPain           0.0       3.0
RestingBP          94.0     200.0
Cholesterol       141.0     564.0
FastingBS           0.0       1.0
RestingECG          0.0       2.0
MaxHR              71.0     202.0
ExerciseAngina      0.0       1.0
Oldpeak             0.0       6.2
Slope               0.0       2.0
MajorVessels        0.0       3.0
Thal                0.0       2.0

Hasil normalisasi training (5 baris pertama):


      Age  Sex  ChestPain  RestingBP  Cholesterol  FastingBS  RestingECG  \
0  0.8723  1.0     1.0000     0.3396       0.4279        0.0         1.0   
1  0.8085  0.0     0.6667     0.1981       1.0000        0.0         1.0   
2  0.5957  1.0     0.3333     0.2830       0.2837        0.0         0.0   
3  0.7447  1.0     1.0000     0.3208       0.2884        0.0         0.0   
4  0.9574  0.0     0.3333     0.2453       0.3026        0.0         1.0   

    MaxHR  ExerciseAngina  Oldpeak  Slope  MajorVessels  Thal  
0  0.2901             0.0   0.3871    0.5        1.0000   0.0  
1  0.6794             0.0   0.2581    0.5        0.0000   1.0  
2  0.5344             0.0   0.0484    0.0        0.0000   1.0  
3  0.2595             1.0   0.0323    0.5        0.3333   1.0  
4  0.3817             1.0   0.0323    0.0        0.3333   0.0  

Hasil normalisasi test (5 baris pertama):
       Age  Sex  ChestPain  RestingBP  Cholesterol  FastingBS  RestingECG  \
6   0.5745  1.0     0.6667     0.3396  

## Nomor 3 — Klasifikasi k-NN dan Decision Tree

Model yang digunakan:
- k-NN dengan k = 1
- k-NN dengan k = 5
- k-NN dengan k = 9
- Decision Tree

Contoh klasifikasi menggunakan fold pertama K-Fold 4 untuk
normalisasi Min-Max, Z-Score, dan Sigmoidal.

In [33]:
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier

# Membuat model baru setiap kali fungsi dipanggil
def buat_model():
    return {
        "k-NN (k=1)": KNeighborsClassifier(
            n_neighbors=1, metric="euclidean"
        ),
        "k-NN (k=5)": KNeighborsClassifier(
            n_neighbors=5, metric="euclidean"
        ),
        "k-NN (k=9)": KNeighborsClassifier(
            n_neighbors=9, metric="euclidean"
        ),
        "Decision Tree": DecisionTreeClassifier(
            random_state=42
        )
    }


# Label mengikuti indeks contoh pembagian pada nomor 2
contoh_train_label = label.iloc[train_idx]
contoh_test_label = label.iloc[test_idx]

hasil_contoh = []

for metode in metode_normalisasi:
    train_normal, test_normal, _ = normalisasi_data(
        contoh_train, contoh_test, metode
    )

    print(f"\n=== Normalisasi {metode} ===")

    perbandingan = pd.DataFrame(
        {"Aktual": contoh_test_label},
        index=contoh_test.index
    )

    for nama_model, model in buat_model().items():
        model.fit(train_normal, contoh_train_label)
        prediksi = model.predict(test_normal)

        perbandingan[nama_model] = prediksi

        jumlah_salah = int(
            (prediksi != contoh_test_label.to_numpy()).sum()
        )
        jumlah_test = len(contoh_test_label)
        error_ratio = jumlah_salah / jumlah_test

        hasil_contoh.append({
            "Normalisasi": metode,
            "Model": nama_model,
            "Jumlah Test": jumlah_test,
            "Jumlah Salah": jumlah_salah,
            "Error Ratio": error_ratio,
            "Akurasi (%)": (1 - error_ratio) * 100
        })

    print("Perbandingan aktual dan prediksi (10 baris pertama):")
    print(perbandingan.head(10))

hasil_contoh = pd.DataFrame(hasil_contoh)

print("\n=== Hasil Klasifikasi Contoh Fold Pertama ===")
print(hasil_contoh.to_string(
    index=False,
    formatters={
        "Error Ratio": "{:.4f}".format,
        "Akurasi (%)": "{:.2f}".format
    }
))

print("\nLabel: 0 = Absence, 1 = Presence")


=== Normalisasi Min-Max ===
Perbandingan aktual dan prediksi (10 baris pertama):
    Aktual  k-NN (k=1)  k-NN (k=5)  k-NN (k=9)  Decision Tree
6        1           1           1           1              1
9        1           1           1           1              1
10       0           1           1           1              1
15       0           0           0           0              1
16       1           1           1           1              1
19       0           1           1           1              0
22       0           0           0           0              0
24       0           0           0           0              0
25       0           0           0           0              0
30       1           0           1           1              0

=== Normalisasi Z-Score ===
Perbandingan aktual dan prediksi (10 baris pertama):
    Aktual  k-NN (k=1)  k-NN (k=5)  k-NN (k=9)  Decision Tree
6        1           1           0           1              1
9        1           1        

## Nomor 4 — Validasi Model

Evaluasi dilakukan menggunakan K-Fold 4, K-Fold 10, dan
Leave-One-Out (LOO) untuk setiap kombinasi normalisasi dan model.

Parameter normalisasi dihitung ulang dari training setiap percobaan.
Error ratio dihitung dari total prediksi salah dibagi total prediksi test.

In [34]:
from sklearn.model_selection import KFold, LeaveOneOut

metode_validasi = {
    "K-Fold 4": KFold(
        n_splits=4, shuffle=True, random_state=42
    ),
    "K-Fold 10": KFold(
        n_splits=10, shuffle=True, random_state=42
    ),
    "LOO": LeaveOneOut()
}

catatan_hasil = []
catatan_detail = []

for nama_validasi, pembagi in metode_validasi.items():
    print(f"\n=== {nama_validasi} ===", flush=True)

    # Pembagian yang sama digunakan untuk seluruh model dan normalisasi
    daftar_split = list(pembagi.split(fitur_data))

    for nama_normalisasi in metode_normalisasi:
        total_salah = {nama: 0 for nama in buat_model()}
        total_test = 0

        for nomor, (idx_train, idx_test) in enumerate(
            daftar_split, start=1
        ):
            X_train = fitur_data.iloc[idx_train]
            X_test = fitur_data.iloc[idx_test]
            y_train = label.iloc[idx_train]
            y_test = label.iloc[idx_test]

            # Normalisasi setelah split
            X_train_normal, X_test_normal, _ = normalisasi_data(
                X_train, X_test, nama_normalisasi
            )

            total_test += len(y_test)

            for nama_model, model in buat_model().items():
                model.fit(X_train_normal, y_train)
                prediksi = model.predict(X_test_normal)

                jumlah_salah = int(
                    (prediksi != y_test.to_numpy()).sum()
                )
                total_salah[nama_model] += jumlah_salah

                catatan_detail.append({
                    "Validasi": nama_validasi,
                    "Normalisasi": nama_normalisasi,
                    "Model": nama_model,
                    "Percobaan": nomor,
                    "Jumlah Test": len(y_test),
                    "Jumlah Salah": jumlah_salah,
                    "Error Ratio": jumlah_salah / len(y_test)
                })

        # Setiap data menjadi test tepat satu kali
        assert total_test == len(fitur_data)

        for nama_model, jumlah_salah in total_salah.items():
            error_ratio = jumlah_salah / total_test

            catatan_hasil.append({
                "Validasi": nama_validasi,
                "Normalisasi": nama_normalisasi,
                "Model": nama_model,
                "Jumlah Percobaan": len(daftar_split),
                "Jumlah Test": total_test,
                "Jumlah Salah": jumlah_salah,
                "Error Ratio": error_ratio,
                "Error (%)": error_ratio * 100,
                "Akurasi (%)": (1 - error_ratio) * 100
            })

            print(
                f"{nama_normalisasi} | {nama_model}: "
                f"{jumlah_salah}/{total_test} salah, "
                f"error = {error_ratio * 100:.2f}%",
                flush=True
            )

hasil_validasi = pd.DataFrame(catatan_hasil)
detail_validasi = pd.DataFrame(catatan_detail)

print("\nJumlah kombinasi yang selesai:", len(hasil_validasi))
assert len(hasil_validasi) == 36


=== K-Fold 4 ===
Min-Max | k-NN (k=1): 63/270 salah, error = 23.33%
Min-Max | k-NN (k=5): 52/270 salah, error = 19.26%
Min-Max | k-NN (k=9): 54/270 salah, error = 20.00%
Min-Max | Decision Tree: 72/270 salah, error = 26.67%
Z-Score | k-NN (k=1): 53/270 salah, error = 19.63%
Z-Score | k-NN (k=5): 44/270 salah, error = 16.30%
Z-Score | k-NN (k=9): 44/270 salah, error = 16.30%
Z-Score | Decision Tree: 73/270 salah, error = 27.04%
Sigmoidal | k-NN (k=1): 54/270 salah, error = 20.00%
Sigmoidal | k-NN (k=5): 48/270 salah, error = 17.78%
Sigmoidal | k-NN (k=9): 47/270 salah, error = 17.41%
Sigmoidal | Decision Tree: 73/270 salah, error = 27.04%

=== K-Fold 10 ===
Min-Max | k-NN (k=1): 61/270 salah, error = 22.59%
Min-Max | k-NN (k=5): 53/270 salah, error = 19.63%
Min-Max | k-NN (k=9): 51/270 salah, error = 18.89%
Min-Max | Decision Tree: 66/270 salah, error = 24.44%
Z-Score | k-NN (k=1): 61/270 salah, error = 22.59%
Z-Score | k-NN (k=5): 49/270 salah, error = 18.15%
Z-Score | k-NN (k=9): 46/

## Nomor 5 — Perbandingan Error Ratio

Menampilkan hasil seluruh kombinasi normalisasi, model klasifikasi,
dan metode validasi.

Error ratio = total prediksi salah / total prediksi test.

In [35]:
print("=== Hasil Seluruh Percobaan ===")

print(hasil_validasi.to_string(
    index=False,
    formatters={
        "Error Ratio": "{:.4f}".format,
        "Error (%)": "{:.2f}".format,
        "Akurasi (%)": "{:.2f}".format
    }
))

# Tabel ringkas: error persentase setiap metode validasi
tabel_error = hasil_validasi.pivot(
    index=["Normalisasi", "Model"],
    columns="Validasi",
    values="Error (%)"
).reindex(columns=["K-Fold 4", "K-Fold 10", "LOO"])

print("\n=== Perbandingan Error (%) ===")
print(tabel_error.round(2).to_string())

# Tampilkan seluruh kombinasi yang memiliki error minimum
# pada masing-masing metode validasi, termasuk jika seri
print("\n=== Error Terkecil per Metode Validasi ===")

for nama_validasi in ["K-Fold 4", "K-Fold 10", "LOO"]:
    kelompok = hasil_validasi[
        hasil_validasi["Validasi"] == nama_validasi
    ]

    minimum_salah = kelompok["Jumlah Salah"].min()
    terbaik = kelompok[
        kelompok["Jumlah Salah"] == minimum_salah
    ]

    print(f"\n{nama_validasi}:")
    print(terbaik[
        ["Normalisasi", "Model", "Jumlah Salah",
        "Error Ratio", "Error (%)", "Akurasi (%)"]
    ].to_string(
        index=False,
        formatters={
            "Error Ratio": "{:.4f}".format,
            "Error (%)": "{:.2f}".format,
            "Akurasi (%)": "{:.2f}".format
        }
    ))

# Simpan hasil agar mudah digunakan dalam laporan
hasil_validasi.to_csv("hasil_validasi_week7.csv", index=False)
detail_validasi.to_csv("detail_validasi_week7.csv", index=False)

print("\nHasil disimpan ke:")
print("- hasil_validasi_week7.csv")
print("- detail_validasi_week7.csv")

=== Hasil Seluruh Percobaan ===
 Validasi Normalisasi         Model  Jumlah Percobaan  Jumlah Test  Jumlah Salah Error Ratio Error (%) Akurasi (%)
 K-Fold 4     Min-Max    k-NN (k=1)                 4          270            63      0.2333     23.33       76.67
 K-Fold 4     Min-Max    k-NN (k=5)                 4          270            52      0.1926     19.26       80.74
 K-Fold 4     Min-Max    k-NN (k=9)                 4          270            54      0.2000     20.00       80.00
 K-Fold 4     Min-Max Decision Tree                 4          270            72      0.2667     26.67       73.33
 K-Fold 4     Z-Score    k-NN (k=1)                 4          270            53      0.1963     19.63       80.37
 K-Fold 4     Z-Score    k-NN (k=5)                 4          270            44      0.1630     16.30       83.70
 K-Fold 4     Z-Score    k-NN (k=9)                 4          270            44      0.1630     16.30       83.70
 K-Fold 4     Z-Score Decision Tree             